# Final QCNN: Rare Visual Anomaly Detection

This is the final, presentation-oriented notebook for the QCNN direction.

The real-world motivation is **rare visual anomaly detection** in settings where failures are uncommon, labels are expensive, and missing a positive case is costly. Two natural examples are:

- **Industrial inspection:** semiconductor wafers, battery electrodes, composite materials, weld scans, surface micro-cracks.
- **Medical imaging:** rare tumor patterns, subtle lesion screening, low-prevalence pathology detection, abnormal tissue texture screening.

The quantum model is a compact **QCNN-style QNN**. It is not a raw image classifier. The intended pipeline is:

1. A classical imaging front-end extracts compact texture/radiomics features from a patch.
2. A structured quantum circuit learns local-to-global feature interactions.
3. The model acts as a high-recall screener for rare visual anomalies.

The careful presentation claim is:

> We are not proving quantum advantage. We are demonstrating a plausible adoption pathway for structured QNNs in rare visual anomaly detection, where ordinary accuracy is misleading and the relevant signal may live in coupled texture features under label scarcity.

## 1. What Is the Classically Hard Problem?

For this project, **classically hard** should not be presented as a theorem about image classification. The difficulty is practical and statistical:

- **Rare positives:** most samples are normal, so a model can look accurate while missing every anomaly.
- **Label scarcity:** true defects or rare pathologies are expensive to label and may have few positive examples.
- **Coupled texture/radiomics features:** anomalies may depend on relationships between features, not one obvious pixel.
- **Asymmetric cost:** a missed defect or missed pathology can be much worse than a false alarm routed to a second inspection step.

Why QCNN? A QCNN-style circuit has local interactions, pooling-like information concentration, and a final readout. That structure is more defensible for visual-feature screening than a large random variational circuit.

## 2. Imports

The notebook uses a fixed seed so the presentation numbers are reproducible.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms.optimizers import COBYLA
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.neural_networks import SamplerQNN

SEED = 8398
DATA_SEED = SEED + 506
np.random.seed(SEED)

## 3. Synthetic Proxy Dataset

This is a synthetic proxy, not a real manufacturing or medical dataset.

Each sample has four compact visual features. In an industrial setting these could be texture/orientation features from inspection. In a medical imaging setting they could be radiomics-style features from a lesion or tissue patch.

The label is created from a hidden coupled feature score. By changing the threshold, we can create:

- a **balanced benchmark** with about 50 percent anomalies,
- an **imbalanced deployment setting** with about 8 percent anomalies.

In [ ]:
def coupled_texture_score(X):
    """Hidden nonlinear visual-feature score used to define anomalies.

    The label depends on feature interactions, not a single obvious threshold.
    This is meant to proxy subtle coupled texture/radiomics behavior.
    """
    return (
        np.sin(X[:, 0]) * np.cos(X[:, 1])
        + np.sin(X[:, 2] + X[:, 3])
        + 0.25 * np.cos(X[:, 0] - X[:, 2])
    )


def generate_visual_anomaly_features(n_samples=260, anomaly_rate=0.08, seed=DATA_SEED):
    """Generate compact visual features and anomaly labels."""
    rng = np.random.default_rng(seed)
    X = rng.uniform(0.0, 2 * np.pi, size=(n_samples, 4))
    scores = coupled_texture_score(X)
    threshold = np.quantile(scores, 1.0 - anomaly_rate)
    y = (scores >= threshold).astype(int)
    return X, y, scores, threshold


def latents_to_texture_images(X, size=8, seed=SEED):
    """Render feature values as small grayscale texture patches for visualization."""
    rng = np.random.default_rng(seed)
    grid = np.linspace(0, 2 * np.pi, size)
    rr, cc = np.meshgrid(grid, grid, indexing="ij")
    images = []

    for x in X:
        img = (
            0.50
            + 0.16 * np.sin(rr + x[0])
            + 0.13 * np.cos(cc + x[1])
            + 0.10 * np.sin(rr + cc + x[2])
            + 0.08 * np.cos(rr - cc + x[3])
        )
        img += rng.normal(0.0, 0.025, size=(size, size))
        images.append(np.clip(img, 0.0, 1.0))

    return np.array(images)

## 4. Build Balanced and Imbalanced Datasets

Same hidden feature process, two evaluation regimes:

- **Balanced:** useful for method comparison when both classes are well represented.
- **Imbalanced:** closer to deployment, where rare positives make ordinary accuracy dangerous.

In [ ]:
N_SAMPLES = 260

balanced_data = generate_visual_anomaly_features(
    n_samples=N_SAMPLES,
    anomaly_rate=0.50,
    seed=DATA_SEED,
)
imbalanced_data = generate_visual_anomaly_features(
    n_samples=N_SAMPLES,
    anomaly_rate=0.08,
    seed=DATA_SEED,
)

datasets = {
    "Balanced benchmark": balanced_data,
    "Imbalanced deployment": imbalanced_data,
}

for name, (X_data, y_data, scores, threshold) in datasets.items():
    print(name)
    print("  clean:", int((y_data == 0).sum()), "anomaly:", int((y_data == 1).sum()), "rate:", round(float(y_data.mean()), 3))
    print("  threshold:", round(float(threshold), 3))

## 5. Presentation Plot: Balanced vs Imbalanced

This plot explains why balanced results and deployment results can tell different stories.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.5), sharey=True)
for ax, (name, (_, y_data, _, _)) in zip(axes, datasets.items()):
    counts = [(y_data == 0).sum(), (y_data == 1).sum()]
    bars = ax.bar(["normal", "anomaly"], counts, color=["#4C78A8", "#E45756"])
    ax.set_title(name)
    ax.set_ylabel("samples")
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2, count + 2, str(int(count)), ha="center")
fig.suptitle("Balanced benchmark vs rare-event deployment")
fig.tight_layout()
plt.show()

## 6. Presentation Plot: Example Patches

These images are only visualization aids. The model receives compact visual features, matching a realistic image -> features -> classifier pipeline.

In [ ]:
def show_selected_textures(X_data, y_data, scores, title, n_normal=6, n_anomaly=6):
    images = latents_to_texture_images(X_data, size=8, seed=SEED)
    normal_idx = np.flatnonzero(y_data == 0)[:n_normal]
    anomaly_idx = np.flatnonzero(y_data == 1)[:n_anomaly]
    order = np.concatenate([normal_idx, anomaly_idx])

    fig, axes = plt.subplots(2, 6, figsize=(9, 3.2))
    axes = np.ravel(axes)
    for ax, idx in zip(axes, order):
        ax.imshow(images[idx], cmap="gray_r", vmin=0, vmax=1)
        label = "ANOM" if y_data[idx] else "normal"
        ax.set_title(f"{label}\nscore={scores[idx]:.2f}", fontsize=8, color="crimson" if y_data[idx] else "black")
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    plt.show()

X_imb, y_imb, scores_imb, _ = imbalanced_data
show_selected_textures(X_imb, y_imb, scores_imb, "Example visual texture patches")

## 7. QCNN-Style Circuit

The circuit is the same for both regimes. It has only four qubits and ten trainable parameters.

Structure:

1. encode four compact visual features,
2. apply local pair interactions,
3. concentrate pair information with pooling-like gates,
4. perform final interaction and read out qubit 0.

In [ ]:
def build_qcnn_style_circuit(n_qubits=4):
    """Build a compact QCNN-style classifier circuit."""
    if n_qubits != 4:
        raise ValueError("This demo circuit is written for exactly four qubits.")

    x = ParameterVector("x", n_qubits)
    theta = ParameterVector("theta", 10)
    qc = QuantumCircuit(n_qubits)

    # Feature encoding: one compact visual/radiomics feature per qubit.
    for q in range(n_qubits):
        qc.ry(x[q], q)

    # Local convolution-like block on pair (0, 1).
    qc.cx(0, 1)
    qc.ry(theta[0], 1)
    qc.rz(theta[1], 0)
    qc.cx(1, 0)
    qc.ry(theta[2], 0)

    # Shared local convolution-like block on pair (2, 3).
    qc.cx(2, 3)
    qc.ry(theta[0], 3)
    qc.rz(theta[1], 2)
    qc.cx(3, 2)
    qc.ry(theta[2], 2)

    # Pooling-like interactions: concentrate pair information into qubits 0 and 2.
    qc.cx(1, 0)
    qc.ry(theta[3], 0)
    qc.cz(1, 0)
    qc.cx(3, 2)
    qc.ry(theta[4], 2)
    qc.cz(3, 2)

    # Final interaction and readout preparation.
    qc.cx(0, 2)
    qc.ry(theta[5], 2)
    qc.rz(theta[6], 0)
    qc.cx(2, 0)
    qc.ry(theta[7], 0)
    qc.ry(theta[8], 2)
    qc.cz(0, 2)
    qc.ry(theta[9], 0)

    return qc, list(x), list(theta)


def readout_q0(measured_integer):
    """Class label is the measured value of qubit 0."""
    return measured_integer & 1

qcnn_circuit, input_params, weight_params = build_qcnn_style_circuit()
print("qubits:", qcnn_circuit.num_qubits)
print("trainable weights:", len(weight_params))
print("circuit depth:", qcnn_circuit.depth())
qcnn_circuit.draw(output="mpl")

## 8. Evaluation Helpers

The imbalanced setting uses every available anomaly plus a controlled number of normal samples for QNN fitting. This makes the QNN behave more like a screening model.

In [ ]:
def metric_row(dataset_name, model_name, y_true, y_pred, scores=None):
    row = {
        "dataset": dataset_name,
        "model": model_name,
        "ordinary_accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "anomaly_f1": f1_score(y_true, y_pred, zero_division=0),
        "anomaly_recall": ((y_pred == 1) & (y_true == 1)).sum() / max(1, (y_true == 1).sum()),
        "false_alarm_rate": ((y_pred == 1) & (y_true == 0)).sum() / max(1, (y_true == 0).sum()),
    }
    if scores is not None:
        row["average_precision"] = average_precision_score(y_true, scores)
        row["roc_auc"] = roc_auc_score(y_true, scores)
    else:
        row["average_precision"] = np.nan
        row["roc_auc"] = np.nan
    return row


def print_row(row):
    print(
        f"{row['dataset']:<23s} {row['model']:<28s} "
        f"acc={row['ordinary_accuracy']:.3f} "
        f"bal={row['balanced_accuracy']:.3f} "
        f"recall={row['anomaly_recall']:.3f} "
        f"false_alarm={row['false_alarm_rate']:.3f} "
        f"F1={row['anomaly_f1']:.3f} "
        f"AP={row['average_precision']:.3f}"
    )


def qnn_fit_subset(X_fit, y_fit, dataset_name, seed=SEED):
    rng = np.random.default_rng(seed)
    anomaly_idx = np.flatnonzero(y_fit == 1)
    normal_idx = np.flatnonzero(y_fit == 0)

    if y_fit.mean() < 0.20:
        # Rare-event regime: every anomaly plus three times as many normal samples.
        n_normal = min(len(normal_idx), len(anomaly_idx) * 3)
        chosen = anomaly_idx.tolist() + rng.choice(normal_idx, size=n_normal, replace=False).tolist()
    else:
        # Balanced regime: balanced subset for stable QNN fitting.
        n_each = min(len(anomaly_idx), len(normal_idx), 36)
        chosen = rng.choice(anomaly_idx, size=n_each, replace=False).tolist()
        chosen += rng.choice(normal_idx, size=n_each, replace=False).tolist()

    rng.shuffle(chosen)
    print(dataset_name, "QNN fit subset clean/anomaly:", int((y_fit[chosen] == 0).sum()), int((y_fit[chosen] == 1).sum()))
    return X_fit[chosen], y_fit[chosen]

## 9. Run Both Experiments

For each dataset we train:

- all-normal/all-clean baseline,
- class-weighted RBF SVM,
- class-weighted random forest,
- QCNN-style QNN.

In [ ]:
rows = []
predictions = {}

for dataset_name, (X_data, y_data, scores_data, threshold) in datasets.items():
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X_data,
        y_data,
        test_size=0.35,
        random_state=SEED,
        stratify=y_data,
    )
    X_fit, X_cal, y_fit, y_cal = train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.25,
        random_state=SEED,
        stratify=y_train_full,
    )

    print()
    print(dataset_name)
    print("fit/test clean/anomaly:", int((y_fit == 0).sum()), int((y_fit == 1).sum()), "/", int((y_test == 0).sum()), int((y_test == 1).sum()))

    # Baseline: predict no anomalies.
    all_normal = np.zeros_like(y_test)
    row = metric_row(dataset_name, "All-normal baseline", y_test, all_normal)
    rows.append(row)
    print_row(row)

    # Classical baselines.
    classical_models = {
        "RBF SVM, class-weighted": Pipeline([
            ("scale", StandardScaler()),
            ("model", SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=SEED)),
        ]),
        "Random Forest, weighted": RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=SEED,
        ),
    }

    for model_name, model in classical_models.items():
        model.fit(X_fit, y_fit)
        pred = model.predict(X_test)
        model_scores = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
        row = metric_row(dataset_name, model_name, y_test, pred, scores=model_scores)
        rows.append(row)
        print_row(row)

    # QCNN-style QNN.
    X_qnn_fit, y_qnn_fit = qnn_fit_subset(X_fit, y_fit, dataset_name)
    sampler_qnn = SamplerQNN(
        circuit=qcnn_circuit,
        sampler=StatevectorSampler(seed=SEED),
        input_params=input_params,
        weight_params=weight_params,
        interpret=readout_q0,
        output_shape=2,
    )
    rng = np.random.default_rng(SEED)
    qcnn_classifier = NeuralNetworkClassifier(
        neural_network=sampler_qnn,
        optimizer=COBYLA(maxiter=50),
        initial_point=rng.uniform(-0.2, 0.2, size=len(weight_params)),
    )
    qcnn_classifier.fit(X_qnn_fit, y_qnn_fit)
    qcnn_pred = qcnn_classifier.predict(X_test)
    qcnn_scores = qcnn_classifier.predict_proba(X_test)[:, 1]
    row = metric_row(dataset_name, "QCNN-style QNN", y_test, qcnn_pred, scores=qcnn_scores)
    rows.append(row)
    print_row(row)

    predictions[dataset_name] = {
        "y_test": y_test,
        "all_normal": all_normal,
        "qcnn_pred": qcnn_pred,
        "qcnn_scores": qcnn_scores,
    }

## 10. Final Metric Table

This is the table to use in speaker notes.

In [ ]:
print("Final metric table")
print("-" * 132)
print(f"{'dataset':<23s} {'model':<28s} {'acc':>7s} {'bal_acc':>8s} {'recall':>8s} {'false_alarm':>12s} {'F1':>7s} {'AP':>7s}")
for row in rows:
    print(
        f"{row['dataset']:<23s} "
        f"{row['model']:<28s} "
        f"{row['ordinary_accuracy']:7.3f} "
        f"{row['balanced_accuracy']:8.3f} "
        f"{row['anomaly_recall']:8.3f} "
        f"{row['false_alarm_rate']:12.3f} "
        f"{row['anomaly_f1']:7.3f} "
        f"{row['average_precision']:7.3f}"
    )

## 11. Presentation Plot: Balanced vs Imbalanced Results

The message is nuanced:

- On the balanced benchmark, classical RBF SVM is strong and the QCNN is competitive.
- On the imbalanced deployment setting, ordinary accuracy is misleading; the QCNN behaves like a high-recall screener.

In [ ]:
plot_models = ["All-normal baseline", "RBF SVM, class-weighted", "Random Forest, weighted", "QCNN-style QNN"]
plot_metrics = ["balanced_accuracy", "anomaly_recall", "false_alarm_rate"]
metric_labels = ["balanced accuracy", "anomaly recall", "false alarm rate"]
colors = ["#4C78A8", "#E45756", "#F58518"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for ax, dataset_name in zip(axes, datasets.keys()):
    subset = [row for row in rows if row["dataset"] == dataset_name]
    x_axis = np.arange(len(plot_models))
    width = 0.23
    for offset, metric, label, color in zip([-width, 0, width], plot_metrics, metric_labels, colors):
        vals = [next(row for row in subset if row["model"] == model)[metric] for model in plot_models]
        ax.bar(x_axis + offset, vals, width, label=label, color=color)
    ax.set_title(dataset_name)
    ax.set_xticks(x_axis)
    ax.set_xticklabels(plot_models, rotation=25, ha="right")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.25)
axes[0].set_ylabel("score")
axes[1].legend(loc="upper right")
fig.suptitle("Balanced benchmark vs rare-event deployment metrics")
fig.tight_layout()
plt.show()

## 12. Presentation Plot: Confusion Matrices for Imbalanced Deployment

This is the clearest plot for the final pitch: the all-normal baseline looks accurate but misses every anomaly; the QCNN catches the anomaly class at the cost of false alarms.

In [ ]:
imb = predictions["Imbalanced deployment"]
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.8))

ConfusionMatrixDisplay.from_predictions(
    imb["y_test"],
    imb["all_normal"],
    display_labels=["normal", "anomaly"],
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title("All-normal baseline")

ConfusionMatrixDisplay.from_predictions(
    imb["y_test"],
    imb["qcnn_pred"],
    display_labels=["normal", "anomaly"],
    ax=axes[1],
    colorbar=False,
)
axes[1].set_title("QCNN-style QNN")

fig.tight_layout()
plt.show()

## 13. Presentation Plot: QCNN Precision-Recall Curves

Precision-recall is more informative than accuracy when anomalies are rare.

In [ ]:
plt.figure(figsize=(6.4, 4.4))
for dataset_name, pred_data in predictions.items():
    precision, recall, _ = precision_recall_curve(pred_data["y_test"], pred_data["qcnn_scores"])
    ap = average_precision_score(pred_data["y_test"], pred_data["qcnn_scores"])
    plt.plot(recall, precision, marker=".", label=f"{dataset_name} AP={ap:.3f}")
plt.xlabel("anomaly recall")
plt.ylabel("precision")
plt.title("QCNN precision-recall curves")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 14. Final Interpretation

Slide-ready wording:

> We tested the same QCNN-style QNN in two regimes. In the balanced benchmark, classical methods are strong and the QCNN is competitive. In the rare-event deployment setting, ordinary accuracy becomes misleading: predicting everything normal looks good but misses every anomaly. The QCNN behaves as a high-recall screening model, trading false alarms for better anomaly capture.

Generic use-case framing:

- Industrial: micro-cracks, wafer defects, battery electrode defects, weld anomalies.
- Medical: rare lesion screening, abnormal tissue texture, low-prevalence pathology triage.

Limitations:

- Synthetic proxy data.
- Simulated QCNN, not hardware.
- No proven quantum advantage.
- Needs stronger classical baselines and real datasets.

Honest final claim:

> Structured QNNs may be useful as compact high-recall screeners for rare visual anomalies when labels are scarce and the signal depends on coupled texture/radiomics features.

## References and AI Tools Disclosure

This notebook was drafted with OpenAI Codex/GPT-5 assistance in the local hackathon workspace. The team should treat the code and results as AI-assisted prototypes and verify claims before presenting them.

AI-assisted notebooks in this `main_challenge` folder:

- `01_defect_classical_hardness_audit.ipynb`
- `02_quantum_kernel_teacher_defects.ipynb`
- `03_projected_quantum_kernel_features.ipynb`
- `04_qnn_vs_kernel_bakeoff.ipynb`
- `05_qcnn_industrial_microdefects.ipynb`
- `06_data_reuploading_qnn_microdefects.ipynb`
- `07_qnn_kernel_pivot_scoreboard.ipynb`
- `08_imbalanced_rare_defect_qnn.ipynb`
- `09_normal_only_anomaly_detection.ipynb`
- `10_heat_exchanger_network_qubo_qaoa.ipynb`
- `11_Final_QCNN_rare_defect_detection.ipynb`

References used for the quantum-ML and optimization direction:

- Cong, Choi, and Lukin, "Quantum convolutional neural networks," Nature Physics 15, 1273-1278 (2019): https://www.nature.com/articles/s41567-019-0648-8
- Perez-Salinas et al., "Data re-uploading for a universal quantum classifier," Quantum 4, 226 (2020): https://doi.org/10.22331/q-2020-02-06-226 and https://arxiv.org/abs/1907.02085
- McClean et al., "Barren plateaus in quantum neural network training landscapes," Nature Communications 9, 4812 (2018): https://www.nature.com/articles/s41467-018-07090-4
- Havlicek et al., "Supervised learning with quantum-enhanced feature spaces," Nature 567, 209-212 (2019): https://www.nature.com/articles/s41586-019-0980-2
- Scholkopf et al., "Estimating the Support of a High-Dimensional Distribution," Neural Computation 13(7), 1443-1471 (2001): https://doi.org/10.1162/089976601750264965
- Qiskit Machine Learning `SamplerQNN` documentation: https://qiskit-community.github.io/qiskit-machine-learning/stubs/qiskit_machine_learning.neural_networks.SamplerQNN.html